# 01 - Preparación de datos

Este notebook deja listo el dataset base para las siguientes etapas. La idea es no trabajar directamente sobre los CSV originales en cada notebook, sino generar tablas intermedias en formato Parquet.

En esta parte se revisa dónde está montado el dataset en Kaggle, se cargan los archivos principales de Instacart y se construyen dos tablas útiles: el historial de compras (`prior_items`) y la última orden etiquetada (`train_items`).


## Verificación del entorno

Primero reviso que el notebook esté corriendo en Kaggle y que exista la carpeta `/kaggle/input`. Esto ayuda a detectar rápido si el dataset fue agregado correctamente como input.


In [2]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\n¿Existe /kaggle/input?")
print(Path("/kaggle/input").exists())

print("\nContenido de /kaggle:")
print(os.listdir("/kaggle") if Path("/kaggle").exists() else "No existe /kaggle")

print("\nContenido de /kaggle/input:")
print(os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else "No existe /kaggle/input")

Current working directory:
/kaggle/working

¿Existe /kaggle/input?
True

Contenido de /kaggle:
['src', 'lib', 'input', 'huggingface', 'nbdev', 'working']

Contenido de /kaggle/input:
['datasets']


## Localización del dataset

Kaggle monta los datasets dentro de `/kaggle/input`. En esta celda reviso el nombre real de la carpeta para evitar errores de ruta.


In [3]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

for folder in INPUT.iterdir():
    print(folder)

/kaggle/input/datasets


## Ruta del dataset

Defino la ruta base del dataset de Instacart. Si Kaggle cambia el nombre de la carpeta o si el input se agregó de otra forma, esta es la única línea que habría que ajustar.


In [4]:
DATASET_DIR = Path("/kaggle/input/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset")

## Carga de archivos CSV

Cargo los seis archivos principales del dataset. En este notebook todavía no hago modelado; solo dejo los datos unidos y en un formato más cómodo para usarlos después.


In [5]:
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset")
WORKING = Path("/kaggle/working")

orders = pd.read_csv(DATASET_DIR / "orders.csv")
prior = pd.read_csv(DATASET_DIR / "order_products__prior.csv")
train = pd.read_csv(DATASET_DIR / "order_products__train.csv")
products = pd.read_csv(DATASET_DIR / "products.csv")
aisles = pd.read_csv(DATASET_DIR / "aisles.csv")
departments = pd.read_csv(DATASET_DIR / "departments.csv")

## Revisión rápida de `orders`

Muestro las primeras filas para confirmar que las columnas clave están presentes: `order_id`, `user_id`, `eval_set` y `order_number`.


In [6]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


## Tamaño de las tablas

Esta revisión sirve para confirmar que todos los archivos cargaron completos y para tener una idea del volumen antes de hacer joins.


In [7]:
print("orders:", orders.shape)
print("prior:", prior.shape)
print("train:", train.shape)
print("products:", products.shape)
print("aisles:", aisles.shape)
print("departments:", departments.shape)

orders: (3421083, 7)
prior: (32434489, 4)
train: (1384617, 4)
products: (49688, 4)
aisles: (134, 2)
departments: (21, 2)


## Distribución de `eval_set`

El campo `eval_set` separa el historial (`prior`), la última orden etiquetada (`train`) y el test original de Kaggle. Para esta prueba, `train` se usará como ground truth final.


In [8]:
orders["eval_set"].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

## Catálogo de productos

Uno `products`, `aisles` y `departments` para tener en una sola tabla el nombre del producto, el pasillo y el departamento. Esto se usa tanto en ranking como en embeddings.


In [9]:
product_catalog = (
    products
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

product_catalog.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


## Historial de compras enriquecido

`order_products__prior.csv` no trae directamente el `user_id`, por eso lo uno con `orders`. También agrego la información del catálogo para tener nombre, aisle y department en cada compra histórica.


In [10]:
prior_items = (
    prior
    .merge(
        orders[["order_id", "user_id", "order_number", "eval_set", "order_dow", "order_hour_of_day"]],
        on="order_id",
        how="left"
    )
    .merge(product_catalog, on="product_id", how="left")
)

prior_items.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,eval_set,order_dow,order_hour_of_day,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,3,prior,5,9,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,3,prior,5,9,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,3,prior,5,9,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,3,prior,5,9,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,3,prior,5,9,Natural Sweetener,17,13,baking ingredients,pantry


## Última orden etiquetada

Repito el mismo proceso para `order_products__train.csv`. Esta tabla queda como referencia para la evaluación final del ranking.


In [11]:
train_items = (
    train
    .merge(
        orders[["order_id", "user_id", "order_number", "eval_set", "order_dow", "order_hour_of_day"]],
        on="order_id",
        how="left"
    )
    .merge(product_catalog, on="product_id", how="left")
)

train_items.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,eval_set,order_dow,order_hour_of_day,product_name,aisle_id,department_id,aisle,department
0,1,49302,1,1,112108,4,train,4,10,Bulgarian Yogurt,120,16,yogurt,dairy eggs
1,1,11109,2,1,112108,4,train,4,10,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16,other creams cheeses,dairy eggs
2,1,10246,3,0,112108,4,train,4,10,Organic Celery Hearts,83,4,fresh vegetables,produce
3,1,49683,4,0,112108,4,train,4,10,Cucumber Kirby,83,4,fresh vegetables,produce
4,1,43633,5,1,112108,4,train,4,10,Lightly Smoked Sardines in Olive Oil,95,15,canned meat seafood,canned goods


## Guardado en Parquet

Guardo las tablas procesadas en `/kaggle/working`. Luego estos archivos se usan como input en los notebooks siguientes.


In [12]:
orders.to_parquet(WORKING / "orders.parquet", index=False)
prior_items.to_parquet(WORKING / "prior_items.parquet", index=False)
train_items.to_parquet(WORKING / "train_items.parquet", index=False)
product_catalog.to_parquet(WORKING / "product_catalog.parquet", index=False)

## Verificación de outputs

Finalmente reviso que los Parquet se hayan creado correctamente. Después de esto se debe hacer `Save Version` en Kaggle para que otro notebook pueda usar estos outputs.


In [13]:
import os

os.listdir("/kaggle/working")

['train_items.parquet',
 'prior_items.parquet',
 'orders.parquet',
 'product_catalog.parquet',
 '__notebook__.ipynb']